# 🚀 Google Colab Remote Development via SSH

### 💻 Use Google Colab Resources

This notebook provides a secure and convenient way to connect a **local VS Code environment** to a **Google Colab runtime** using **SSH and a public TCP tunnel**.

The primary purpose of this setup is to combine the **development experience of VS Code** with the **computational resources available in Google Colab**, including GPU, CPU, RAM, and temporary storage.

### 🔗 How It Works

```text
┌──────────────────┐
│   Local Machine  │
│      VS Code     │
└────────┬─────────┘
         │ SSH
         ▼
┌──────────────────┐
│   Public Tunnel  │
│      Pinggy      │
└────────┬─────────┘
         │ TCP
         ▼
┌──────────────────┐
│   Google Colab   │
│   Remote Runtime │
│                  │
│  CPU • RAM • GPU │
│  Storage • Python│
└──────────────────┘

```
**⚠️ Important Notes**

This is a temporary development environment.

The Colab runtime and its resources are temporary and may be terminated or reset when the runtime disconnects. The free Pinggy tunnel is also temporary and expires after its allowed session duration.

## Install SSH server

In [25]:
!apt-get update -qq
!apt-get install -y openssh-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openssh-server is already the newest version (1:9.6p1-3ubuntu13.19).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


## Create colab user

In [26]:
import subprocess

USERNAME = "colab"
PASSWORD = "colab123"

# Create user if it doesn't exist
subprocess.run(
    ["useradd", "-m", "-s", "/bin/bash", USERNAME],
    stderr=subprocess.DEVNULL
)

# Set password
subprocess.run(
    ["chpasswd"],
    input=f"{USERNAME}:{PASSWORD}\n",
    text=True,
    check=True
)

print(f"User created: {USERNAME}")
print(f"Password: {PASSWORD}")

User created: colab
Password: colab123


## Configure SSH

In [27]:
ssh_config = """\
Port 22
ListenAddress 0.0.0.0

Protocol 2

PermitRootLogin no

PasswordAuthentication yes
KbdInteractiveAuthentication yes
PubkeyAuthentication no

UsePAM no

AllowUsers colab

MaxAuthTries 10
LoginGraceTime 60

X11Forwarding no
AllowTcpForwarding yes
"""

with open("/etc/ssh/sshd_config", "w") as f:
    f.write(ssh_config)

print(ssh_config)

Port 22
ListenAddress 0.0.0.0

Protocol 2

PermitRootLogin no

PasswordAuthentication yes
KbdInteractiveAuthentication yes
PubkeyAuthentication no

UsePAM no

AllowUsers colab

MaxAuthTries 10
LoginGraceTime 60

X11Forwarding no
AllowTcpForwarding yes



## Prepare SSH server

In [28]:
!mkdir -p /run/sshd
!chmod 755 /run/sshd

# Generate host keys if required
!ssh-keygen -A

# Validate SSH configuration
!sshd -t

print("SSH configuration is valid.")

SSH configuration is valid.


## Start SSH server

In [29]:
# First kill any old SSH server:
!pkill sshd || true

In [30]:
# Then start it:
!mkdir -p /run/sshd
!/usr/sbin/sshd

## Verify SSH server

In [31]:
!ss -lntp | grep ':22'

# You should see something similar to:
# LISTEN 0 128 0.0.0.0:22

LISTEN 0      128          0.0.0.0:22         0.0.0.0:*    users:(("sshd",pid=5834,fd=3))         


## Verify the colab user

In [ ]:
!id colab

## Test SSH locally: SKIP

In [33]:
!ssh \
    -o StrictHostKeyChecking=no \
    -o UserKnownHostsFile=/dev/null \
    -o PreferredAuthentications=password \
    -o PubkeyAuthentication=no \
    colab@127.0.0.1 \
    'whoami'

Permission denied, please try again.
Permission denied, please try again.
colab@127.0.0.1: Permission denied (password,keyboard-interactive).


### If you don't want an interactive password prompt

**Install sshpass:**

In [34]:
!apt-get install -y sshpass -qq

Selecting previously unselected package sshpass.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../sshpass_1.09-1_amd64.deb ...
Unpacking sshpass (1.09-1) ...
Setting up sshpass (1.09-1) ...
Processing triggers for man-db (2.12.0-4build2) ...


## This confirms that Colab SSH is working before Pinggy is involved.

In [35]:
!sshpass -p 'colab123' ssh \
    -o StrictHostKeyChecking=no \
    -o UserKnownHostsFile=/dev/null \
    -o PreferredAuthentications=password \
    -o PubkeyAuthentication=no \
    colab@127.0.0.1 \
    'whoami'

colab


## Install Pinggy

In [ ]:
!ssh -T \
    -o StrictHostKeyChecking=no \
    -o ServerAliveInterval=30 \
    -o ServerAliveCountMax=3 \
    -p 443 \
    -R0:127.0.0.1:22 \
    tcp@free.pinggy.io

Allocated port 9 for remote forward to 127.0.0.1:22
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
tcp://xdlwr-34-46-137-118.run.pinggy-free.link:36889
RB: 796189, SB: 3730387, TC: 1, AC: 1               

**You should get something like:**
```
Allocated port 3 for remote forward to 127.0.0.1:22

You are not authenticated.
Your tunnel will expire in 60 minutes.

tcp://mykwn-34-46-137-118.run.pinggy-free.link:41917
```

**The important part is:**
```
Host:
mykwn-34-46-137-118.run.pinggy-free.link

Port:
41917
```

**EXAMPLE SSH CONFIG:**
```Host colab
    HostName xdlwr-34-46-137-118.run.pinggy-free.link # Need To Update
    Port 36889 # Need To Update
    User colab
    PreferredAuthentications password
    PubkeyAuthentication no
    KbdInteractiveAuthentication yes```

---

---

# Configure VS Code

**Install Extension:**
```VS Code - Extensions - Remote SSH```

**open:**
`C:\Users\<name>\.ssh\config`

**Put:**
```
Host colab
    HostName mykwn-34-46-137-118.run.pinggy-free.link
    Port 41917
    User colab
    PreferredAuthentications password
    PubkeyAuthentication no
    KbdInteractiveAuthentication yes
```

**Choose:**
- Linux
- Continue
- Password

---

🎉 **Congratulations! You’re Connected to Google Colab! 🚀**

Your **VS Code is now connected directly to the Google Colab machine via Remote SSH**.

💻 **VS Code** → 🔐 **SSH** → 🌐 **Pinggy Tunnel** → ☁️ **Google Colab**

You can now code locally in VS Code while using **Colab’s GPU, RAM, CPU, and storage resources**. 🔥

**Happy coding! 🚀🐍⚡**
